# Kobol VCO grey-box: colab playground
Train the grey-box VCO model (saw core -> tri fold + 2 comparators -> lowpass -> chain HP) on the Kobol dataset, with knobs and plots for basically everything.

**How to use:** Runtime -> Run all. CPU is fine (GPU doesn't really help, the batch is tiny). Change the knobs in section 1 and re-run from there.
Rough timing on colab CPU: data prep ~3-5 min the first time (cached after), training 1500 steps ~1-2 min.

In [ ]:
#@title 0. setup + download dataset (~460 MB, only first time)
!pip -q install soundfile ipywidgets

import os
DATA_DIR = "kobol"  # @param {type:"string"}
if not os.path.isdir(f"{DATA_DIR}/audios"):
    !curl -sL -o kobol.zip https://www.kaggle.com/api/v1/datasets/download/bringmethetxcos/kobolrsf-vco
    !unzip -qo kobol.zip -d {DATA_DIR}
else:
    print("data already there, skipping download")
print(len(os.listdir(f"{DATA_DIR}/audios")), "wav files")  # should be 437

## 1. Knobs
Everything you might want to poke at. Re-run this cell + everything below after changing stuff.

In [ ]:
#@title knobs
# --- training ---
STEPS = 1500  # @param {type:"slider", min:100, max:6000, step:100}
LR = 3e-3  # @param {type:"number"}
LR_DROP = 0.3  # @param {type:"number"}
# lr gets multiplied by LR_DROP at 50% and 75% of the steps
HIDDEN = 64  # @param [32, 64, 128, 192] {type:"raw"}
DEPTH = 3  # @param {type:"slider", min:1, max:5, step:1}
SEED = 0  # @param {type:"integer"}
LOG_EVERY = 50  # @param {type:"slider", min:10, max:500, step:10}
LOGMAG_W = 0.002  # @param {type:"number"}
# ^ weight of the log-magnitude term in the loss (helps the quiet high harmonics)

# --- held-out split ---
SCHEME = "rand5"  # @param ["rand5", "col", "row"]
FOLD = 0  # @param {type:"slider", min:0, max:4, step:1}
# rand5 = random 5 folds, col = whole waveform columns held out, row = whole CV rows held out (col/row only have 4 folds)

# --- grey-box structure (ablation switches) ---
USE_TRI = True  # @param {type:"boolean"}
USE_COMP_B = True  # @param {type:"boolean"}
USE_LOWPASS = True  # @param {type:"boolean"}
USE_CHAIN_HP = True  # @param {type:"boolean"}
USE_LP2 = True  # @param {type:"boolean"}
USE_SLEW = True  # @param {type:"boolean"}
USE_RESET = False  # @param {type:"boolean"}
L1 = 2e-3  # @param {type:"number"}
# defaults above = the "final" variant from RESULTS.md

# --- baseline ---
TRAIN_BLACKBOX = True  # @param {type:"boolean"}
BB_STEPS = 4000  # @param {type:"slider", min:500, max:8000, step:500}

# --- data prep ---
K = 256  # @param {type:"integer"}
FORCE_RECOMPUTE = False  # @param {type:"boolean"}

if SCHEME != "rand5" and FOLD > 3:
    print("col/row only have folds 0-3, using 3"); FOLD = 3
FLAGS = dict(tri=USE_TRI, compB=USE_COMP_B, lp=USE_LOWPASS, hp=USE_CHAIN_HP, lp2=USE_LP2, slew=USE_SLEW, reset=USE_RESET, l1=L1)
print(FLAGS)

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, soundfile as sf, glob, time
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from IPython.display import display, clear_output, Audio
torch.set_num_threads(2)  # colab cpu has 2 cores, more threads was slower for me lol
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
SR = 44100

## 2. Load audio + estimate f0
437 files = 19 waveform settings (w = 0..18, cvw = 0.5 V steps) x 23 pitch CV steps (n = 21..43, cvf = 2.64 + 0.33(n-21) V).
f0: autocorrelation for a coarse guess, then FFT peak refinement. The thesis labels are wrong at the ends so we re-estimate everything.

In [ ]:
D = {}
for p in glob.glob(f"{DATA_DIR}/audios/*.wav"):
    a, b = map(int, os.path.basename(p)[:-4].split("_"))
    x, sr = sf.read(p)
    D[(a, b)] = x
assert sr == SR, sr
print(len(D), "files, e.g.", D[(0, 21)].shape)

def trim(x):
    # skip the silence at the start + 1500 samples of settling
    if x.ndim > 1: x = x[:, 0]
    m = np.abs(x).max(); i = np.argmax(np.abs(x) > 0.05 * m)
    return x[i + 1500:]

def f0_est(x):
    x = x - x.mean(); n = len(x)
    X = np.fft.rfft(x, 2 * n); ac = np.fft.irfft(np.abs(X) ** 2)[:n]; ac /= ac[0]
    ac = ac * n / (n - np.arange(n))  # unbias
    z = np.argmax(ac < 0) if (ac < 0).any() else 2
    lo = max(z, 2); hi = min(int(0.2 * SR), n // 2)
    L = lo + np.argmax(ac[lo:hi]); mx = ac[L]
    # take the smallest lag peak that's close to the max, otherwise octave errors everywhere
    peaks = [i for i in range(lo + 1, hi - 1) if ac[i] > ac[i - 1] and ac[i] >= ac[i + 1] and ac[i] > 0.85 * mx]
    if peaks: L = peaks[0]
    k = max(1, int((n // 2) / L) - 1); Lk = int(round(L * k))  # look k periods out, more precise
    w = max(2, int(0.02 * L) + 2)
    j = Lk - w + np.argmax(ac[Lk - w:Lk + w + 1])
    a, b, c = ac[j - 1], ac[j], ac[j + 1]
    d = 0.5 * (a - c) / (a - 2 * b + c) if (a - 2 * b + c) != 0 else 0
    return SR * k / (j + d), mx

def fft_refine(y, coarse, tol=0.06):
    y = y - y.mean(); N = len(y)
    Y = np.abs(np.fft.rfft(y * np.hanning(N), N * 8)); fr = np.fft.rfftfreq(N * 8, 1 / SR)
    m = (fr > coarse * (1 - tol)) & (fr < coarse * (1 + tol)); i = np.where(m)[0][np.argmax(Y[m])]
    a, b, c = np.log(Y[i - 1:i + 2] + 1e-12); d = 0.5 * (a - c) / (a - 2 * b + c)
    return (i + d) * (fr[1] - fr[0])

In [ ]:
if os.path.exists("f0b.csv") and not FORCE_RECOMPUTE:
    f0df = pd.read_csv("f0b.csv"); print("loaded cached f0b.csv")
else:
    t0 = time.time()
    rows = []
    for (a, b), x in D.items():
        f, q = f0_est(trim(x)); rows.append((a, b, f))
        # if q < 0.5: print("low acf peak??", a, b, q)
    d = pd.DataFrame(rows, columns=["w", "n", "f0"])
    med = d.groupby("n").f0.median().to_dict()
    # top 2 CV steps: acf gets confused up there, just take the biggest fft peak above 1k
    for n in [42, 43]:
        y = trim(D[(0, n)]); y = y - y.mean(); Y = np.abs(np.fft.rfft(y)); fr = np.fft.rfftfreq(len(y), 1 / SR)
        med[n] = fr[np.argmax(Y * (fr > 1000))]
    out = []
    for (a, b), x in D.items():
        y = trim(x)
        out.append((a, b, fft_refine(y, med[b]) if med[b] > 40 else d[(d.w == a) & (d.n == b)].f0.item()))
    f0df = pd.DataFrame(out, columns=["w", "n", "f0"]).sort_values(["w", "n"]).reset_index(drop=True)
    f0df.to_csv("f0b.csv", index=False)
    print("f0 done in %.0fs" % (time.time() - t0))
f0df.head()

In [ ]:
#@title plot: f0 vs pitch CV, and detune between waveforms
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
cmap = plt.cm.viridis
for w in range(19):
    s = f0df[f0df.w == w]
    ax[0].semilogy(2.64 + 0.33 * (s.n - 21), s.f0, ".-", color=cmap(w / 18), lw=0.8, ms=3)
ax[0].set_xlabel("pitch CV (V)"); ax[0].set_ylabel("f0 (Hz)"); ax[0].set_title("f0 for all 19 waveform settings")
piv = f0df.pivot(index="n", columns="w", values="f0")
cents = 1200 * np.log2(piv.div(piv.median(axis=1), axis=0))
im = ax[1].imshow(cents.values, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-30, vmax=30,
                  extent=[-0.5, 18.5, 20.5, 43.5])
plt.colorbar(im, ax=ax[1], label="cents vs row median")
ax[1].set_xlabel("waveform w"); ax[1].set_ylabel("CV step n"); ax[1].set_title("waveform knob detunes the pitch")
plt.tight_layout(); plt.show()

## 3. Harmonics
256 harmonics per file by projecting onto an integer number of periods. Phases rotated so H1 is a sine, so every file is aligned the same way. Above 18 kHz is masked out (nothing useful there).

In [ ]:
if os.path.exists("harm.npz") and not FORCE_RECOMPUTE:
    z = np.load("harm.npz", allow_pickle=True); C, Mk, meta = z["C"], z["M"], z["meta"].astype(float)
    if C.shape[1] != K: print("K changed, set FORCE_RECOMPUTE=True and rerun")
    print("loaded cached harm.npz")
else:
    t0 = time.time(); rows = []; Cl = []; Ml = []
    k = np.arange(1, K + 1)
    for _, r in f0df.iterrows():
        w, n, f0 = int(r.w), int(r.n), r.f0
        y = trim(D[(w, n)]); y = y - y.mean()
        P = SR / f0; L = int(int(len(y) / P) * P); y = y[:L]  # whole number of periods
        t = np.arange(L) / SR; win = np.hanning(L)
        c = np.zeros(K, complex)
        for i in range(0, K, 32):  # chunked bc 256 x 40k complex blew up the ram on my laptop
            kk = k[i:i + 32, None]
            c[i:i + 32] = 2 * (np.exp(-2j * np.pi * kk * f0 * t) * (y * win)).sum(1) / win.sum()
        rot = np.angle(c[0]) + np.pi / 2
        c = c * np.exp(-1j * k * rot)
        rows.append((w, n, f0, w * 0.5, 2.64 + (n - 21) * 0.33)); Cl.append(c); Ml.append(k * f0 < 18000)
    meta = pd.DataFrame(rows, columns=["w", "n", "f0", "cvw", "cvf"]).values.astype(float)
    C, Mk = np.array(Cl), np.array(Ml)
    np.savez("harm.npz", C=C, M=Mk, meta=meta)
    print("harmonics done in %.0fs" % (time.time() - t0))

W, Nn, F0 = meta[:, 0].astype(int), meta[:, 1].astype(int), meta[:, 2]
CVF = 2.64 + (Nn - 21) * 0.33; CVW = W * 0.5
key = {(W[i], Nn[i]): i for i in range(len(W))}
# print(C.shape, Mk.sum(1)[:10])
print(C.shape, "| valid harmonics per file: min", Mk.sum(1).min(), "max", Mk.sum(1).max())

def cycle(c, m=None, n=512):
    # one period from harmonics
    if m is not None: c = c * m
    ph = np.arange(n) / n; k = np.arange(1, len(c) + 1)
    return np.real((c[:, None] * np.exp(2j * np.pi * k[:, None] * ph)).sum(0))

def render(f, c, n=SR):
    t = np.arange(n) / SR; y = np.zeros(n); k = np.arange(1, len(c) + 1); v = k * f < 18000
    for kk, cc in zip(k[v], c[v]): y += np.abs(cc) * np.cos(2 * np.pi * kk * f * t + np.angle(cc))
    return y

In [ ]:
#@title plot: one cycle of every waveform setting + harmonic spectra
PLOT_N = 30  # @param {type:"slider", min:21, max:43, step:1}
fig, ax = plt.subplots(4, 5, figsize=(13, 8), sharex=True)
for w in range(19):
    a = ax.flat[w]; i = key[(w, PLOT_N)]
    a.plot(cycle(C[i], Mk[i]), lw=1); a.set_title(f"w={w} ({CVW[i]:.1f} V)", fontsize=8); a.set_yticks([])
ax.flat[19].axis("off")
plt.suptitle(f"recorded cycles at CV step n={PLOT_N} ({F0[key[(0, PLOT_N)]]:.0f} Hz)"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for w in [0, 4, 9, 12, 16]:
    i = key[(w, PLOT_N)]; kk = np.arange(1, K + 1)[Mk[i]]
    ax[0].plot(kk, 20 * np.log10(np.abs(C[i][Mk[i]]) + 1e-6), ".-", ms=2, lw=0.7, label=f"w={w}")
ax[0].set_xscale("log"); ax[0].set_xlabel("harmonic"); ax[0].set_ylabel("dB"); ax[0].legend(); ax[0].set_title("harmonic magnitudes")
H = np.array([20 * np.log10(np.abs(C[key[(w, PLOT_N)]][:40]) + 1e-6) for w in range(19)])
im = ax[1].imshow(H.T, aspect="auto", origin="lower", cmap="magma", vmin=-80, vmax=0)
plt.colorbar(im, ax=ax[1], label="dB"); ax[1].set_xlabel("waveform w"); ax[1].set_ylabel("harmonic - 1"); ax[1].set_title("first 40 harmonics vs waveform knob")
plt.tight_layout(); plt.show()

## 4. Train / held-out split

In [ ]:
def folds(scheme):
    if scheme == "rand5":
        p = np.random.RandomState(0).permutation(len(W)); return [np.sort(p[i::5]) for i in range(5)]
    if scheme == "col":  # interior waveform columns, interleaved
        return [np.where((W >= 1) & (W <= 17) & (W % 4 == r))[0] for r in range(4)]
    if scheme == "row":  # interior CV rows, interleaved
        return [np.where((Nn >= 22) & (Nn <= 42) & (Nn % 4 == r))[0] for r in range(4)]

VA = folds(SCHEME)[FOLD]
TR = np.setdiff1d(np.arange(len(W)), VA)
print(f"{SCHEME} fold {FOLD}: {len(TR)} train, {len(VA)} held out")

grid = np.zeros((23, 19)); grid[Nn[VA] - 21, W[VA]] = 1
plt.figure(figsize=(4.5, 4)); plt.imshow(grid, origin="lower", cmap="Greys", extent=[-0.5, 18.5, 20.5, 43.5], aspect="auto")
plt.xlabel("waveform w"); plt.ylabel("CV step n"); plt.title("black = held out"); plt.show()

## 5. Pitch: physical law
f0 = 2^(a + b(cvf) + c(cvw) + d(cvw x cvf) + offset[n]). The per-step offsets are shared across all waveforms, they come from the CV source (MCV24) not the VCO.
Fit on train only, test on held-out.

In [ ]:
NI = Nn - 21
def phys(p, cvf, cvw, ni):
    a, b, c, d = p[:4]; dl = p[4:]
    if np.ndim(ni) == 0 and not float(ni).is_integer():  # in-between steps (for the explorer): interp offsets
        off = np.interp(ni, np.arange(23), dl)
    else:
        off = dl[np.asarray(ni, int)]
    return 2 ** (a + b * (cvf - 6.27) + c * (cvw - 4.5) + d * (cvw - 4.5) * (cvf - 6.27) + off)

def pres(p, idx):
    r = 1200 * np.log2(phys(p, CVF[idx], CVW[idx], NI[idx]) / F0[idx])
    dl = p[4:][2:21]; kk = np.arange(19) - 9
    return np.r_[r, 300 * dl.sum(), 300 * (dl * kk).sum()]  # offsets: no mean / no slope, otherwise not identifiable

p0 = np.r_[np.log2(400), 1.5, 0, 0, np.zeros(23)]
PP = least_squares(pres, p0, args=(TR,)).x
pc = lambda idx: 1200 * np.log2(phys(PP, CVF[idx], CVW[idx], NI[idx]) / F0[idx])
interior = lambda idx: idx[(Nn[idx] >= 22) & (Nn[idx] <= 42)]
print("oct/V = %.4f   detune c=%.5f d=%.5f" % (PP[1], PP[2], PP[3]))
print("pitch rms cents  train %.2f | held-out %.2f | held-out interior %.2f" % (
    np.sqrt(np.mean(pc(TR) ** 2)), np.sqrt(np.mean(pc(VA) ** 2)), np.sqrt(np.mean(pc(interior(VA)) ** 2))))
# print(np.round(1200 * PP[4:], 1))

In [ ]:
#@title plot: pitch errors
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].bar(np.arange(21, 44), 1200 * PP[4:], color="C1"); ax[0].set_xlabel("CV step n"); ax[0].set_ylabel("cents")
ax[0].set_title("per-step CV offsets (from the MCV24)")
err = np.full((23, 19), np.nan); allc = pc(np.arange(len(W))); err[NI, W] = allc
im = ax[1].imshow(err, origin="lower", cmap="RdBu_r", vmin=-10, vmax=10, aspect="auto", extent=[-0.5, 18.5, 20.5, 43.5])
ax[1].scatter(W[VA], Nn[VA], s=6, c="k", marker="x", lw=0.5)
plt.colorbar(im, ax=ax[1], label="cents"); ax[1].set_xlabel("w"); ax[1].set_ylabel("n"); ax[1].set_title("pitch error (x = held out)")
ax[2].hist(pc(TR), 40, alpha=0.6, label="train", density=True); ax[2].hist(pc(VA), 40, alpha=0.6, label="held out", density=True)
ax[2].set_xlim(-15, 15); ax[2].set_xlabel("cents"); ax[2].legend(); ax[2].set_title("pitch error histogram")
plt.tight_layout(); plt.show()

## 6. Models
**Grey-box (GB):** an MLP maps (cvf, cvw) -> 8 numbers: stage gains (tri, saw, comp A, comp B), 2 comparator thresholds, tri phase, lowpass cutoff.
The harmonics are then computed in closed form (so no aliasing): saw + folded triangle + two comparator pulses -> 1-pole lowpass (+ optional 2nd pole) -> recording-chain highpass.

**Black-box (BB):** MLP straight to 512 numbers (Re/Im of the 256 harmonics). The v2 baseline.

In [ ]:
def mlp(i, h, o, d):
    L = [nn.Linear(i, h), nn.SiLU()]
    for _ in range(d - 1): L += [nn.Linear(h, h), nn.SiLU()]
    return nn.Sequential(*L, nn.Linear(h, o))

k = torch.arange(1, K + 1, dtype=torch.float32); k64 = k.double()
Nt = 8192; ph = np.arange(Nt) / Nt
TRI0 = torch.tensor((2 * np.fft.fft(1 - 2 * np.abs(2 * ph - 1)) / Nt)[1:K + 1].astype(np.complex64))
E = lambda phi: torch.exp(-2j * np.pi * k * phi)       # delay by phi periods
E64 = lambda phi: torch.exp(-2j * np.pi * k64 * phi)

def pwl(bps, dsl):
    # periodic piecewise-linear wave from breakpoints + slope changes -> fourier coeffs
    return (2 * (dsl[:, :, None] * E64(bps[:, :, None])).sum(1) / ((2j * np.pi * k64) ** 2)).to(torch.complex64)

OUT_NAMES = ["A_tri", "A_saw", "A_compA", "thr_A", "A_compB", "thr_B", "tri_phase", "log_fc"]

class GB(nn.Module):
    def __init__(s, fl, hidden=64, depth=3):
        super().__init__(); s.fl = fl; s.net = mlp(2, hidden, 8, depth)
        s.lfh = nn.Parameter(torch.tensor(np.log(5.0)))            # chain highpass (Hz)
        s.lTr = nn.Parameter(torch.tensor(np.log(2e-6)))           # core reset time
        s.lsl = nn.Parameter(torch.tensor([np.log(2e-6), np.log(2e-6)], dtype=torch.float32))  # comparator edge times
        s.lfc2 = nn.Parameter(torch.tensor(np.log(20000.)))        # 2nd lowpass pole
        with torch.no_grad():
            # start near a sane waveform, random init just gave noise for ages
            s.net[-1].bias[:] = torch.tensor([0.1, 0.1, 0.1, 0.0, 0.1, 1.0, 0.0, np.log(15000.)]); s.net[-1].weight.mul_(0.1)

    def forward(s, x, f, parts=False):
        o = s.net(x).float(); fl = s.fl; f = f.float()[:, None]; B = len(f)
        At, As, A1, t1, A2, t2, pt, lfc = [o[:, i:i + 1] for i in range(8)]
        th1, th2 = torch.tanh(t1), torch.tanh(t2)
        if fl.get("reset"):
            r = torch.clamp(f.double() * torch.exp(s.lTr.double()), 1e-4, 0.3); s1 = 2 / (1 - r); s2 = -2 / r; z = torch.zeros_like(r)
            SAW = pwl(torch.cat([z, 1 - r], 1), torch.cat([s1 - s2, s2 - s1], 1))
            pa = (1 - r) / 2; pb = 1 - r / 2
            TRI = pwl(torch.cat([z, pa, 1 - r, pb], 1), torch.cat([2 * s1 - 2 * s2, -4 * s1, -2 * s2 + 2 * s1, 4 * s2], 1))
            rf = r.float(); cross = lambda th: ((1 - rf) * (1 + th) / 2, (1 - rf) + rf * (1 - th) / 2)
        else:
            SAW = (2j / (np.pi * k)).expand(B, K); TRI = TRI0.expand(B, K)
            cross = lambda th: ((1 + th) / 2, torch.ones_like(th))
        def comp(th):
            p1, p2 = cross(th); a, b = E(p1), E(p2)
            if fl.get("slew"):
                e = f * torch.exp(s.lsl); a = a * torch.sinc(k * e[:, 0:1]); b = b * torch.sinc(k * e[:, 1:2])
            return 2 * (a - b) / (1j * np.pi * k)
        P = {"saw": As * SAW, "compA": A1 * comp(th1)}
        if fl.get("tri", True): P["tri"] = At * TRI * E(0.1 * torch.tanh(pt))
        if fl.get("compB", True): P["compB"] = A2 * comp(th2)
        fk = k * f
        H = torch.ones_like(fk, dtype=torch.complex64)
        if fl.get("lp", True): H = H / (1 + 1j * fk / torch.exp(lfc))
        if fl.get("lp2"): H = H / (1 + 1j * fk / torch.exp(s.lfc2))
        if fl.get("hp", True): fh = torch.exp(s.lfh); H = H * (1j * fk / fh) / (1 + 1j * fk / fh)
        c = sum(P.values()) * H
        rot = torch.exp(-1j * k * (torch.angle(c[:, 0]) + np.pi / 2)[:, None])  # same alignment as the data
        if parts: return {n: v * H * rot for n, v in P.items()}, o
        return c * rot

    def globals_(s):
        return dict(chainHP_Hz=float(torch.exp(s.lfh)), lp2_Hz=float(torch.exp(s.lfc2)),
                    slew_us=[round(v, 2) for v in (torch.exp(s.lsl) * 1e6).tolist()], reset_us=float(torch.exp(s.lTr)) * 1e6)

class BB(nn.Module):
    def __init__(s, hidden=128, depth=3): super().__init__(); s.net = mlp(2, hidden, 2 * K, depth)
    def forward(s, x, f): o = s.net(x).float(); return torch.complex(o[:, :K], o[:, K:])

normx = lambda cvf, cvw: np.stack([(np.asarray(cvf) - 6.27) / 3.63, (np.asarray(cvw) - 4.5) / 4.5], -1)
Xall = torch.tensor(normx(CVF, CVW), dtype=torch.float32)
Ct = torch.tensor(C.astype(np.complex64)); Mt = torch.tensor(Mk.astype(np.float32)); Ft = torch.tensor(F0.astype(np.float32))

def lossf(c, i):
    m = Mt[i]; mse = ((torch.abs(c - Ct[i]) ** 2) * m).sum(1).mean()
    lm = ((torch.log(torch.abs(c) + 1e-4) - torch.log(torch.abs(Ct[i]) + 1e-4)).abs() * m).sum(1).mean() / K
    return mse + LOGMAG_W * lm

def hsnr(c, ref, m): return 10 * np.log10((np.abs(ref * m) ** 2).sum() / (np.abs((c - ref) * m) ** 2).sum())
def lsd(c, ref, m): return np.sqrt(np.mean((20 * np.log10((np.abs(c[m]) + 1e-4) / (np.abs(ref[m]) + 1e-4))) ** 2))

## 7. Train
Live plot: loss (train vs held-out), held-out median SNR, gradient norm, and the learned global params.

In [ ]:
def train(model, steps, lr, opt_kind="adam", wd=0.0, l1=0.0, name="model"):
    torch.manual_seed(SEED)
    opt = torch.optim.Adam(model.parameters(), lr) if opt_kind == "adam" else torch.optim.AdamW(model.parameters(), lr, weight_decay=wd)
    tr, va = torch.tensor(TR), torch.tensor(VA)
    hist = dict(step=[], train=[], val=[], snr_va=[], snr_tr=[], gnorm=[], glob=[])
    gn_buf = []; t0 = time.time()
    for it in range(steps):
        opt.zero_grad()
        L = lossf(model(Xall[tr], Ft[tr]), tr)
        if l1: L = L + l1 * model.net(Xall[tr])[:, [0, 1, 2, 4]].abs().mean()  # push unused stages to 0
        L.backward()
        gn_buf.append(float(torch.nn.utils.clip_grad_norm_(model.parameters(), 1e9)))  # 1e9 = not actually clipping, just reading the norm
        opt.step()
        if it in (steps // 2, 3 * steps // 4): opt.param_groups[0]["lr"] *= LR_DROP
        if it % LOG_EVERY == 0 or it == steps - 1:
            with torch.no_grad():
                c = model(Xall, Ft); lv = float(lossf(c[va], va)); c = c.numpy()
            hist["step"].append(it); hist["train"].append(L.item()); hist["val"].append(lv)
            hist["snr_va"].append(np.median([hsnr(c[j], C[j], Mk[j]) for j in VA]))
            hist["snr_tr"].append(np.median([hsnr(c[j], C[j], Mk[j]) for j in TR]))
            hist["gnorm"].append(np.mean(gn_buf)); gn_buf = []
            if hasattr(model, "globals_"): hist["glob"].append(model.globals_())
            live_plot(hist, name, it, steps, time.time() - t0)
    return hist

def live_plot(h, name, it, steps, el):
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 4 if h["glob"] else 3, figsize=(16, 3.3))
    ax[0].semilogy(h["step"], h["train"], label="train"); ax[0].semilogy(h["step"], h["val"], "--", label="held out")
    ax[0].set_title("loss"); ax[0].legend()
    ax[1].plot(h["step"], h["snr_tr"], label="train"); ax[1].plot(h["step"], h["snr_va"], "--", label="held out")
    ax[1].set_title("median harmonic SNR (dB)"); ax[1].legend()
    ax[2].semilogy(h["step"], h["gnorm"]); ax[2].set_title("grad norm")
    if h["glob"]:
        g = h["glob"]; a = ax[3]; a2 = a.twinx()
        a.plot(h["step"], [x["chainHP_Hz"] for x in g], "C0", label="chain HP (Hz)")
        a2.plot(h["step"], [x["lp2_Hz"] / 1000 for x in g], "C1", label="2nd pole (kHz)")
        a.set_title("learned globals"); a.legend(loc="upper left", fontsize=7); a2.legend(loc="lower right", fontsize=7); a2.grid(False)
    for a in ax: a.set_xlabel("step")
    plt.suptitle(f"{name}: step {it+1}/{steps}  ({el:.0f}s)  held-out SNR {h['snr_va'][-1]:.2f} dB", fontsize=10)
    plt.tight_layout(); plt.show(); plt.close(fig)  # without close it leaks a figure every log step

In [ ]:
gb = GB(FLAGS, HIDDEN, DEPTH)
print(sum(p.numel() for p in gb.parameters()), "params")
hist_gb = train(gb, STEPS, LR, l1=FLAGS["l1"], name="grey-box")
print("final:", gb.globals_())

In [ ]:
#@title train the black-box baseline (skip with TRAIN_BLACKBOX=False)
if TRAIN_BLACKBOX:
    bb = BB()
    hist_bb = train(bb, BB_STEPS, 2e-3, opt_kind="adamw", wd=0.2, name="black-box")
else:
    bb = None; print("skipping black box")

## 8. Results on the held-out settings
Compared with a dumb baseline: average the harmonics of the neighbouring training recordings.

In [ ]:
with torch.no_grad():
    c_gb = gb(Xall, Ft).numpy()
    c_bb = bb(Xall, Ft).numpy() if bb is not None else None

TRS = set(TR)
def neighbour_avg(j):
    nb = [key[(W[j] + dw, Nn[j])] for dw in (-1, 1) if (W[j] + dw, Nn[j]) in key and key[(W[j] + dw, Nn[j])] in TRS]
    if not nb:  # whole column held out -> try rows
        nb = [key[(W[j], Nn[j] + dn)] for dn in (-1, 1) if (W[j], Nn[j] + dn) in key and key[(W[j], Nn[j] + dn)] in TRS]
    if not nb: return np.zeros(K, complex)
    return np.mean([C[i] * Mk[i] for i in nb], 0)
c_nb = np.array([neighbour_avg(j) if j in set(VA) else C[j] for j in range(len(W))])

methods = {"grey-box": c_gb, "neighbour avg": c_nb}
if c_bb is not None: methods["black-box"] = c_bb
family = lambda w: "tri" if w <= 4 else "saw" if w <= 10 else "square" if w <= 13 else "pulse"
register = lambda n: "low" if n <= 27 else ("mid" if n <= 35 else "high")
res = pd.DataFrame([dict(method=mn, j=j, w=W[j], n=Nn[j], fam=family(W[j]), reg=register(Nn[j]),
                         snr=hsnr(c[j], C[j], Mk[j]), lsd=lsd(c[j], C[j], Mk[j]))
                    for mn, c in methods.items() for j in VA])
print(res.groupby("method")[["snr", "lsd"]].median().round(2).to_string())
# res.sort_values("snr").head(10)

In [ ]:
#@title plot: SNR distribution, by family / register, error per harmonic
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
for mn in methods:
    ax[0].hist(res[res.method == mn].snr, bins=np.arange(-5, 40, 1.5), alpha=0.5, label=mn)
ax[0].set_xlabel("held-out SNR (dB)"); ax[0].legend(); ax[0].set_title("SNR distribution")
res.pivot_table(index="fam", columns="method", values="snr", aggfunc="median").loc[["tri", "saw", "square", "pulse"]].plot.bar(ax=ax[1], rot=0)
ax[1].set_title("median SNR by family"); ax[1].set_xlabel("")
res.pivot_table(index="reg", columns="method", values="snr", aggfunc="median").loc[["low", "mid", "high"]].plot.bar(ax=ax[2], rot=0, legend=False)
ax[2].set_title("median SNR by register"); ax[2].set_xlabel("")
kk = np.arange(1, K + 1)
for mn, c in methods.items():
    e = np.abs(c[VA] - C[VA]) / (np.abs(C[VA]).sum(1, keepdims=True) / Mk[VA].sum(1, keepdims=True))
    e = np.where(Mk[VA], e, np.nan)
    ax[3].loglog(kk, np.nanmedian(e, 0), lw=1, label=mn)
ax[3].set_xlabel("harmonic"); ax[3].set_title("median relative error per harmonic"); ax[3].legend()
plt.tight_layout(); plt.show()

In [ ]:
#@title plot: SNR over the whole grid (train AND held out) - see where it struggles
fig, ax = plt.subplots(1, len(methods), figsize=(5 * len(methods), 4))
for a, (mn, c) in zip(np.atleast_1d(ax), methods.items()):
    g = np.full((23, 19), np.nan)
    for j in range(len(W)): g[NI[j], W[j]] = hsnr(c[j], C[j], Mk[j]) if (mn != "neighbour avg" or j in set(VA)) else np.nan
    im = a.imshow(g, origin="lower", cmap="viridis", vmin=0, vmax=30, aspect="auto", extent=[-0.5, 18.5, 20.5, 43.5])
    a.scatter(W[VA], Nn[VA], s=8, c="r", marker="x", lw=0.6)
    a.set_title(f"{mn} SNR (x = held out)"); a.set_xlabel("w"); a.set_ylabel("n")
plt.colorbar(im, ax=ax, label="dB"); plt.show()

In [ ]:
#@title plot: best / median / worst held-out cycles
s = res[res.method == "grey-box"].sort_values("snr").reset_index(drop=True)
pick = [0, 1, len(s) // 2, len(s) - 2, len(s) - 1]
fig, ax = plt.subplots(1, 5, figsize=(17, 3))
for a, r in zip(ax, pick):
    j = int(s.j[r])
    a.plot(cycle(C[j], Mk[j]), "k", lw=1.5, label="recording")
    for mn, c in methods.items():
        if mn != "neighbour avg": a.plot(cycle(c[j], Mk[j]), lw=1, label=mn)
    a.set_title(f"w={W[j]} n={Nn[j]}  GB {s.snr[r]:.1f} dB", fontsize=9); a.set_yticks([])
ax[0].legend(fontsize=7); plt.suptitle("worst  <-  held-out cycles  ->  best"); plt.tight_layout(); plt.show()

## 9. Probe the grey-box
What did the MLP actually learn? Stage gains / thresholds / cutoff along the waveform knob, and what each stage contributes.

In [ ]:
#@title plot: MLP outputs along the waveform knob
PROBE_CVF = 5.5  # @param {type:"slider", min:2.64, max:9.9, step:0.01}
cvw_s = np.linspace(0, 9, 200)
with torch.no_grad():
    xs = torch.tensor(normx(np.full_like(cvw_s, PROBE_CVF), cvw_s), dtype=torch.float32)
    fs = torch.tensor(phys(PP, PROBE_CVF, cvw_s, (PROBE_CVF - 2.64) / 0.33), dtype=torch.float32)
    parts, o = gb(xs, fs, parts=True); o = o.numpy()
fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
for i in [0, 1, 2, 4]: ax[0].plot(cvw_s, o[:, i], label=OUT_NAMES[i])
ax[0].set_title("stage gains"); ax[0].legend(fontsize=7)
ax[1].plot(cvw_s, (1 + np.tanh(o[:, 3])) / 2, label="comp A duty"); ax[1].plot(cvw_s, (1 + np.tanh(o[:, 5])) / 2, label="comp B duty")
ax[1].set_title("comparator duty"); ax[1].legend(fontsize=7); ax[1].set_ylim(0, 1)
ax[2].semilogy(cvw_s, np.exp(o[:, 7]) / 1000); ax[2].set_title("output lowpass fc (kHz)")
for n_, v in parts.items(): ax[3].plot(cvw_s, np.sqrt((np.abs(v.numpy()) ** 2).sum(1) / 2), label=n_)
ax[3].set_title("rms contribution of each stage"); ax[3].legend(fontsize=7)
for a in ax: a.set_xlabel("waveform CV (V)")
plt.suptitle(f"pitch CV = {PROBE_CVF} V"); plt.tight_layout(); plt.show()

In [ ]:
#@title plot: weights per layer
lins = [m for m in gb.net if isinstance(m, nn.Linear)]
fig, ax = plt.subplots(1, len(lins), figsize=(3.4 * len(lins), 2.6))
for i, (a, m) in enumerate(zip(np.atleast_1d(ax), lins)):
    w = m.weight.detach().numpy().ravel(); a.hist(w, 50); a.set_title(f"layer {i} {tuple(m.weight.shape)}\nstd {w.std():.3f}", fontsize=8)
plt.tight_layout(); plt.show()

## 10. Explorer
Turn the knobs, see + hear the model. "nearest recording" is the closest measured setting (it's only exact on the 19x23 grid).

In [ ]:
import ipywidgets as wd
try:
    from google.colab import output; output.enable_custom_widget_manager()
except Exception: pass

def explore(cvf=5.5, cvw=4.0, show_parts=True, play=False):
    f = float(phys(PP, cvf, cvw, (cvf - 2.64) / 0.33))
    with torch.no_grad():
        x = torch.tensor(normx([cvf], [cvw]), dtype=torch.float32); ft = torch.tensor([f])
        c = gb(x, ft).numpy()[0]; parts, o = gb(x, ft, parts=True)
    m = np.arange(1, K + 1) * f < 18000
    j = key[(int(np.clip(round(cvw / 0.5), 0, 18)), int(np.clip(round((cvf - 2.64) / 0.33) + 21, 21, 43)))]
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
    ax[0].plot(cycle(C[j], Mk[j]), "k", lw=2, alpha=0.4, label=f"nearest recording (w={W[j]}, n={Nn[j]})")
    ax[0].plot(cycle(c, m), "C3", lw=1.5, label="grey-box")
    if show_parts:
        for n_, v in parts.items(): ax[0].plot(cycle(v.numpy()[0], m), "--", lw=0.8, label=n_)
    ax[0].legend(fontsize=7); ax[0].set_title(f"f0 = {f:.1f} Hz"); ax[0].set_xlabel("sample in cycle")
    kk = np.arange(1, K + 1)
    ax[1].plot(kk[Mk[j]], 20 * np.log10(np.abs(C[j][Mk[j]]) + 1e-6), "k.", ms=3, alpha=0.5, label="nearest recording")
    ax[1].plot(kk[m], 20 * np.log10(np.abs(c[m]) + 1e-6), "C3", lw=1, label="grey-box")
    ax[1].set_xscale("log"); ax[1].set_ylim(-100, 5); ax[1].set_xlabel("harmonic"); ax[1].set_ylabel("dB"); ax[1].legend(fontsize=7)
    plt.tight_layout(); plt.show()
    print("mlp out:", {n_: round(float(v), 3) for n_, v in zip(OUT_NAMES, o[0])})
    if play:
        y = render(f, c, SR); y = 0.5 * y / np.abs(y).max()
        r = trim(D[(W[j], Nn[j])])[:SR]; r = 0.5 * r / np.abs(r).max()
        print("model:"); display(Audio(y, rate=SR)); print("nearest recording:"); display(Audio(r, rate=SR))

wd.interact(explore,
            cvf=wd.FloatSlider(value=5.5, min=2.64, max=9.9, step=0.01, description="pitch CV", continuous_update=False),
            cvw=wd.FloatSlider(value=4.0, min=0, max=9, step=0.05, description="wave CV", continuous_update=False),
            show_parts=True, play=False);

## 11. Save

In [ ]:
torch.save(gb.state_dict(), f"gb_{SCHEME}_f{FOLD}_s{SEED}.pt")
np.save("pitch_phys_colab.npy", PP)
res.to_csv(f"heldout_{SCHEME}_f{FOLD}.csv", index=False)
print("saved. grab them from the files panel on the left")
# from google.colab import files; files.download(f"gb_{SCHEME}_f{FOLD}_s{SEED}.pt")